In [ ]:
from __future__ import annotations


class AnalyticsService:
    """
    Aggregates learning and product activity from MongoDB.

    All queries are scoped by user_id and, when appropriate,
    project_id to preserve tenant/user isolation.
    """

    def __init__(self, database):
        self.database = database

    # ========================================================
    # PROJECT SUMMARY
    # ========================================================

    async def get_project_summary(
        self,
        *,
        user_id: str,
        project_id: str,
    ) -> dict:

        materials = self.database.collection(
            "materials"
        )

        activities = self.database.collection(
            "activities"
        )

        assessments = self.database.collection(
            "assessments"
        )

        mastery = self.database.collection(
            "mastery"
        )

        material_filter = {
            "project_id": project_id,
            "user_id": user_id,
        }

        total_materials = materials.count_documents(
            material_filter
        )

        # IMPORTANT:
        # DocumentService stores the processing state in
        # processing_status, not status.
        processed_materials = materials.count_documents(
            {
                **material_filter,
                "processing_status": "completed",
            }
        )

        failed_materials = materials.count_documents(
            {
                **material_filter,
                "processing_status": "failed",
            }
        )

        processing_materials = materials.count_documents(
            {
                **material_filter,
                "processing_status": "processing",
            }
        )

        # ----------------------------------------------------
        # TUTOR
        # ----------------------------------------------------

        tutor_interactions = activities.count_documents(
            {
                "project_id": project_id,
                "user_id": user_id,
                "event_type": {
                    "$in": [
                        "tutor_interaction",
                        "tutor_question",
                        "tutor_answer",
                    ]
                },
            }
        )

        # ----------------------------------------------------
        # COMPLETED ASSESSMENTS
        # ----------------------------------------------------

        completed_assessments = list(
            assessments.find(
                {
                    "project_id": project_id,
                    "user_id": user_id,
                    "status": "completed",
                },
                {
                    "_id": 0,
                    "answers": 1,
                    "score": 1,
                    "max_score": 1,
                },
            )
        )

        quizzes_completed = len(
            completed_assessments
        )

        questions_answered = sum(
            len(
                assessment.get(
                    "answers",
                    [],
                )
            )
            for assessment in completed_assessments
        )

        # ----------------------------------------------------
        # AVERAGE SCORE
        # ----------------------------------------------------

        percentages = []

        for assessment in completed_assessments:

            score = assessment.get(
                "score"
            )

            max_score = assessment.get(
                "max_score"
            )

            if (
                score is not None
                and max_score is not None
                and float(max_score) > 0
            ):

                percentage = (
                    float(score)
                    / float(max_score)
                )

                percentages.append(
                    max(
                        0.0,
                        min(
                            1.0,
                            percentage,
                        ),
                    )
                )

        average_score = (
            sum(percentages)
            / len(percentages)
            if percentages
            else None
        )

        # ----------------------------------------------------
        # MASTERY
        # ----------------------------------------------------

        mastery_documents = list(
            mastery.find(
                {
                    "project_id": project_id,
                    "user_id": user_id,
                },
                {
                    "_id": 0,
                    "concept_id": 1,
                    "score": 1,
                    "trend": 1,
                },
            )
        )

        concepts_tracked = len(
            mastery_documents
        )

        concepts_needing_attention = sum(
            1
            for item in mastery_documents
            if item.get("trend")
            == "needs_attention"
        )

        return {
            "project_id": project_id,
            "total_materials": total_materials,
            "processed_materials": processed_materials,
            "failed_materials": failed_materials,
            "processing_materials": processing_materials,
            "tutor_interactions": tutor_interactions,
            "quizzes_completed": quizzes_completed,
            "questions_answered": questions_answered,
            "average_score": average_score,
            "concepts_tracked": concepts_tracked,
            "concepts_needing_attention": (
                concepts_needing_attention
            ),
        }

    # ========================================================
    # ACTIVITY
    # ========================================================

    async def get_activity(
        self,
        *,
        user_id: str,
        project_id: str | None = None,
        limit: int = 50,
    ) -> list[dict]:

        limit = max(
            1,
            min(
                int(limit),
                200,
            ),
        )

        query = {
            "user_id": user_id,
        }

        if project_id is not None:
            query["project_id"] = project_id

        documents = (
            self.database.collection(
                "activities"
            )
            .find(
                query,
                {
                    "_id": 0,
                },
            )
            .sort(
                "created_at",
                -1,
            )
            .limit(limit)
        )

        return list(
            documents
        )
